# FREIGHT-MNet MLP-Residual Experiment

This notebook implements the next experiment after the M0 and M0.5 tabular baselines: **MLP-Residual quantile regression**.

The goal is not to replace the M0.5 LightGBM hybrid baseline immediately. Instead, this notebook tests whether a neural nonlinear model can learn useful residual corrections around a strong train-only OD historical prior. The experiment uses the same supervised FAF OD-year dataset, the same temporal split, and the same target labels as the previous baseline notebooks.

**Main forecasting task**

\[
(o, d, t, x_{od,t}) \rightarrow (\widehat{q}_{25}, \widehat{q}_{50}, \widehat{q}_{75})
\]

where the targets are annual truck travel-time quantiles in minutes:

- `truck_q25`
- `truck_q50`
- `truck_q75`

**Temporal split**

- Train: 2018--2022
- Validation: 2023
- Test: 2024

**Primary scope**

- `east_plus_gulf`

The notebook is written as a complete, modular experiment script. It saves model checkpoints, metrics, predictions, training history, feature metadata, and run configuration.

## 1. Experiment design

This notebook trains and evaluates the following models:

1. **Global train median**  
   A minimum sanity baseline using the train-set global median for each quantile.

2. **OD historical prior**  
   A strong temporal-persistence baseline. For validation and test rows, the prior is the same OD pair's median over train years. For train rows, the prior is leave-one-training-row-out, so a row never uses its own label as its own prior.

3. **MLP raw current features**  
   A neural no-history baseline using only the 64 manifest features.

4. **MLP prior direct**  
   A direct neural model using the 64 manifest features plus historical prior features.

5. **MLP residual current features**  
   A residual model initialized around the OD historical prior, using only the 64 manifest features for correction.

6. **MLP residual prior features**  
   A residual model initialized around the OD historical prior, using both the 64 manifest features and historical prior features for correction.

The direct MLP models use a **cumulative softplus monotone quantile head**:

\[
\widehat{q}_{25}=\operatorname{softplus}(a), \quad
\widehat{q}_{50}=\widehat{q}_{25}+\operatorname{softplus}(b), \quad
\widehat{q}_{75}=\widehat{q}_{50}+\operatorname{softplus}(c).
\]

The residual MLP models use a **residualized monotone head**. The model predicts corrections to the historical-prior location and gaps while preserving monotonicity by construction.

The training objective is:

\[
\mathcal{L} = \operatorname{WeightedPinball}(q_{25},q_{50},q_{75}) + \lambda_{IQR}\operatorname{SmoothL1}(\widehat{IQR}, IQR).
\]

This matches the project design: weighted quantile regression, monotone quantile prediction, and explicit IQR risk modeling.

In [1]:
# ============================================================
# Cell 1: Environment setup and imports
# ============================================================

from __future__ import annotations

import json
import math
import os
import random
import shutil
import sys
import time
import warnings
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

# Avoid optional pandas acceleration modules becoming part of the experiment logic.
# This is helpful on Windows/Anaconda environments where NumPy ABI conflicts can appear.
os.environ.setdefault("PYTHONHASHSEED", "42")
os.environ.setdefault("NUMEXPR_MAX_THREADS", "8")

warnings.filterwarnings("default")

import numpy as np
import pandas as pd

# Disable pandas optional acceleration after import. The notebook does not need these paths.
pd.set_option("compute.use_numexpr", False)
pd.set_option("compute.use_bottleneck", False)

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, Dataset
except Exception as exc:
    raise RuntimeError(
        "PyTorch is required for this MLP-residual notebook. "
        "Please run it in the FREIGHT-MNet CUDA/torch environment."
    ) from exc

print("Python executable:", sys.executable)
print("Python version   :", sys.version)
print("NumPy version    :", np.__version__)
print("Pandas version   :", pd.__version__)
print("PyTorch version  :", torch.__version__)
print("CUDA available   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device      :", torch.cuda.get_device_name(0))

Python executable: E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Scripts\python.exe
Python version   : 3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]
NumPy version    : 2.4.5
Pandas version   : 2.3.3
PyTorch version  : 2.12.0+cu126
CUDA available   : True
CUDA device      : NVIDIA GeForce RTX 4050 Laptop GPU


## 2. Configuration

The configuration cell is the only place where normal users should need to edit paths or hyperparameters.

Default input paths follow the existing project layout:

```text
E:\NetworkOptimization\pythonProject1\Data\08_processed\model_ready\
  freight_mnet_supervised_edges_2018_2024_east_plus_gulf.parquet

E:\NetworkOptimization\pythonProject1\Data\08_processed\model_ready\_metadata\
  freight_mnet_supervised_feature_manifest.json
```

Default outputs are written under:

```text
E:\NetworkOptimization\pythonProject1\Data\10_experiments\mlp_residual_v1_notebook\east_plus_gulf\
```

The experiment uses the same label columns and temporal split as the M0/M0.5 experiments.

In [2]:
# ============================================================
# Cell 2: Experiment configuration
# ============================================================

@dataclass
class ExperimentConfig:
    """Central configuration for the MLP-residual experiment."""

    # Project paths and scope.
    data_root: Path = Path(r"E:\NetworkOptimization\pythonProject1\Data")
    scope: str = "east_plus_gulf"  # Use "all" for the robustness scope.
    run_name: str = "mlp_residual_v1_notebook"
    seed: int = 42
    overwrite: bool = True
    save_models: bool = True

    # Temporal split.
    train_years: Tuple[int, ...] = (2018, 2019, 2020, 2021, 2022)
    val_year: int = 2023
    test_year: int = 2024

    # Supervised target columns.
    label_columns: Tuple[str, ...] = ("truck_q25", "truck_q50", "truck_q75")
    taus: Tuple[float, ...] = (0.25, 0.50, 0.75)

    # Loss weighting. The weight column is not used as a predictive feature.
    use_sample_weight: bool = True
    weight_column: str = "obs_weight_sum"
    weight_clip_min: float = 0.05
    weight_clip_max: float = 20.0

    # MLP architecture and optimization.
    hidden_dim: int = 256
    n_hidden_layers: int = 3
    dropout: float = 0.10
    batch_size: int = 4096
    max_epochs: int = 300
    patience: int = 30
    min_delta: float = 1e-5
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    grad_clip_norm: float = 5.0
    lambda_iqr: float = 0.10
    num_workers: int = 0

    # Model switches.
    run_mlp_raw_current_features: bool = True
    run_mlp_prior_direct: bool = True
    run_mlp_residual_current_features: bool = True
    run_mlp_residual_prior_features: bool = True

    # Calibration is off by default because the M0 experiments showed global validation
    # median calibration can hurt 2024 test performance under temporal shift.
    run_global_val_median_calibration: bool = False

    # Numeric stability.
    positive_epsilon: float = 1e-6
    float_output_precision: int = 6


config = ExperimentConfig()
config

ExperimentConfig(data_root=WindowsPath('E:/NetworkOptimization/pythonProject1/Data'), scope='east_plus_gulf', run_name='mlp_residual_v1_notebook', seed=42, overwrite=True, save_models=True, train_years=(2018, 2019, 2020, 2021, 2022), val_year=2023, test_year=2024, label_columns=('truck_q25', 'truck_q50', 'truck_q75'), taus=(0.25, 0.5, 0.75), use_sample_weight=True, weight_column='obs_weight_sum', weight_clip_min=0.05, weight_clip_max=20.0, hidden_dim=256, n_hidden_layers=3, dropout=0.1, batch_size=4096, max_epochs=300, patience=30, min_delta=1e-05, learning_rate=0.001, weight_decay=0.0001, grad_clip_norm=5.0, lambda_iqr=0.1, num_workers=0, run_mlp_raw_current_features=True, run_mlp_prior_direct=True, run_mlp_residual_current_features=True, run_mlp_residual_prior_features=True, run_global_val_median_calibration=False, positive_epsilon=1e-06, float_output_precision=6)

## 3. Reproducibility and path management

This cell defines utility functions for deterministic seeding and project path resolution.

The notebook intentionally gives the prediction file a model-specific name:

```text
predictions_val_test_mlp_residual.parquet
```

This prevents the M0, M0.5, MLP, and later graph experiments from overwriting each other's prediction files.

In [3]:
# ============================================================
# Cell 3: Reproducibility and path utilities
# ============================================================

@dataclass
class ExperimentPaths:
    """Resolved filesystem paths used by this experiment."""

    data_root: Path
    supervised_path: Path
    manifest_path: Path
    output_dir: Path
    models_dir: Path


def set_global_seed(seed: int) -> None:
    """Set random seeds for Python, NumPy, and PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Deterministic algorithms are not forced because some GPU ops may become very slow.
    torch.backends.cudnn.benchmark = True


def normalize_scope(scope: str) -> str:
    """Normalize scope aliases into the filename suffix used by the data pipeline."""
    scope_clean = scope.strip().lower()
    if scope_clean in {"east_plus_gulf", "east-plus-gulf", "eg", "main"}:
        return "east_plus_gulf"
    if scope_clean in {"all", "all_valid", "national"}:
        return "all"
    raise ValueError(f"Unsupported scope: {scope!r}. Use 'east_plus_gulf' or 'all'.")


def resolve_paths(cfg: ExperimentConfig) -> ExperimentPaths:
    """Resolve input and output paths from the experiment configuration."""
    scope_suffix = normalize_scope(cfg.scope)
    data_root = Path(cfg.data_root)
    model_ready_dir = data_root / "08_processed" / "model_ready"
    supervised_path = model_ready_dir / f"freight_mnet_supervised_edges_2018_2024_{scope_suffix}.parquet"
    manifest_path = model_ready_dir / "_metadata" / "freight_mnet_supervised_feature_manifest.json"
    output_dir = data_root / "10_experiments" / cfg.run_name / scope_suffix
    models_dir = output_dir / "models"
    return ExperimentPaths(
        data_root=data_root,
        supervised_path=supervised_path,
        manifest_path=manifest_path,
        output_dir=output_dir,
        models_dir=models_dir,
    )


def prepare_output_dir(paths: ExperimentPaths, overwrite: bool) -> None:
    """Create the output directory and optionally clear existing files."""
    if paths.output_dir.exists() and overwrite:
        shutil.rmtree(paths.output_dir)
    paths.output_dir.mkdir(parents=True, exist_ok=True)
    paths.models_dir.mkdir(parents=True, exist_ok=True)


set_global_seed(config.seed)
paths = resolve_paths(config)
prepare_output_dir(paths, config.overwrite)

print("Supervised table:", paths.supervised_path)
print("Feature manifest:", paths.manifest_path)
print("Output directory:", paths.output_dir)

Supervised table: E:\NetworkOptimization\pythonProject1\Data\08_processed\model_ready\freight_mnet_supervised_edges_2018_2024_east_plus_gulf.parquet
Feature manifest: E:\NetworkOptimization\pythonProject1\Data\08_processed\model_ready\_metadata\freight_mnet_supervised_feature_manifest.json
Output directory: E:\NetworkOptimization\pythonProject1\Data\10_experiments\mlp_residual_v1_notebook\east_plus_gulf


## 4. Constants and leakage guards

The feature manifest is the source of truth for predictive features. The notebook performs leakage checks before training.

Important rules:

- `truck_q25`, `truck_q50`, and `truck_q75` are labels and must not appear in the predictive feature list.
- FMI aggregation diagnostics such as `n_fmi_county_pairs` and `obs_weight_sum` are not predictive features. They may be used for diagnostics and loss weighting only.
- Historical prior features are constructed explicitly and are clearly named with the `hist_` prefix.

In [4]:
# ============================================================
# Cell 4: Constants and feature leakage guards
# ============================================================

FMI_DIAGNOSTIC_COLUMNS = {
    "n_fmi_county_pairs",
    "obs_weight_sum",
    "input_q25_min",
    "input_q25_max",
    "input_q50_min",
    "input_q50_max",
    "input_q75_min",
    "input_q75_max",
}

HISTORICAL_PRIOR_FEATURE_COLUMNS = [
    "hist_truck_q25",
    "hist_truck_q50",
    "hist_truck_q75",
    "hist_truck_iqr",
    "hist_truck_q75_q50_gap",
    "hist_truck_q50_q25_gap",
    "hist_n_prior_years",
    "hist_has_prior",
]


def load_feature_manifest(manifest_path: Path) -> List[str]:
    """Load predictive feature columns from the feature manifest JSON."""
    if not manifest_path.exists():
        raise FileNotFoundError(f"Feature manifest not found: {manifest_path}")

    with manifest_path.open("r", encoding="utf-8") as f:
        manifest = json.load(f)

    if "feature_columns" not in manifest:
        raise KeyError("The feature manifest must contain a 'feature_columns' key.")

    feature_columns = list(manifest["feature_columns"])
    if not feature_columns:
        raise ValueError("The feature manifest contains an empty feature column list.")

    return feature_columns


def validate_feature_columns(
    feature_columns: Sequence[str],
    label_columns: Sequence[str],
) -> None:
    """Raise an error if the predictive feature list contains leakage-prone columns."""
    label_set = set(label_columns)
    feature_set = set(feature_columns)

    label_overlap = sorted(feature_set.intersection(label_set))
    diagnostic_overlap = sorted(feature_set.intersection(FMI_DIAGNOSTIC_COLUMNS))
    derived_label_like = sorted(
        c for c in feature_columns
        if c.startswith("truck_q") or c.startswith("input_q") or c.endswith("_target")
    )

    problems = []
    if label_overlap:
        problems.append(f"label columns in features: {label_overlap}")
    if diagnostic_overlap:
        problems.append(f"FMI diagnostic columns in features: {diagnostic_overlap}")
    if derived_label_like:
        problems.append(f"label-like derived columns in features: {derived_label_like}")

    if problems:
        raise ValueError("Potential feature leakage detected: " + "; ".join(problems))


manifest_feature_columns = load_feature_manifest(paths.manifest_path)
validate_feature_columns(manifest_feature_columns, config.label_columns)

feature_columns_with_prior = manifest_feature_columns + HISTORICAL_PRIOR_FEATURE_COLUMNS

print(f"Manifest feature count     : {len(manifest_feature_columns)}")
print(f"Feature count with prior   : {len(feature_columns_with_prior)}")
print("First 10 manifest features :", manifest_feature_columns[:10])

Manifest feature count     : 64
Feature count with prior   : 72
First 10 manifest features : ['dest_east_plus_gulf_county_share', 'dest_east_plus_gulf_faf_flag_any_county', 'dest_max_county_centroid_lon', 'dest_min_county_centroid_lon', 'dest_n_counties', 'dest_n_east_plus_gulf_counties', 'has_multimodal_demand', 'has_rail_demand', 'has_truck_demand', 'log1p_tmiles_multimodal']


## 5. Data loading and schema checks

This cell reads the model-ready supervised table and verifies the required columns.

The code infers the origin, destination, and year column names from common candidates. In the current pipeline, the expected columns are:

- `faf_orig`
- `faf_dest`
- `year`

All label columns are converted to numeric floating-point values. The notebook also verifies that the observed labels satisfy the expected monotonicity:

```text
truck_q25 <= truck_q50 <= truck_q75
```

In [5]:
# ============================================================
# Cell 5: Data loading and schema checks
# ============================================================


def infer_column(df: pd.DataFrame, candidates: Sequence[str], purpose: str) -> str:
    """Return the first existing column from a candidate list."""
    for col in candidates:
        if col in df.columns:
            return col
    raise KeyError(f"Could not infer {purpose} column. Tried: {list(candidates)}")


def load_supervised_table(
    supervised_path: Path,
    feature_columns: Sequence[str],
    label_columns: Sequence[str],
) -> Tuple[pd.DataFrame, str, str, str]:
    """Load the supervised OD-year table and run schema validation."""
    if not supervised_path.exists():
        raise FileNotFoundError(f"Supervised parquet file not found: {supervised_path}")

    df = pd.read_parquet(supervised_path).reset_index(drop=True)

    orig_col = infer_column(df, ["faf_orig", "origin_faf", "orig_faf", "origin", "orig"], "origin")
    dest_col = infer_column(df, ["faf_dest", "destination_faf", "dest_faf", "destination", "dest"], "destination")
    year_col = infer_column(df, ["year", "Year", "yr"], "year")

    required_columns = set(feature_columns).union(label_columns).union({orig_col, dest_col, year_col})
    missing = sorted(c for c in required_columns if c not in df.columns)
    if missing:
        raise KeyError(f"The supervised table is missing required columns: {missing[:20]}")

    df[year_col] = pd.to_numeric(df[year_col], errors="raise").astype(int)
    for col in label_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    if df[list(label_columns)].isna().any().any():
        bad_counts = df[list(label_columns)].isna().sum().to_dict()
        raise ValueError(f"Label columns contain missing values: {bad_counts}")

    y = df[list(label_columns)].to_numpy(dtype=np.float64)
    crossing_count = int(np.sum((y[:, 0] > y[:, 1]) | (y[:, 1] > y[:, 2])))
    if crossing_count > 0:
        raise ValueError(f"Observed labels contain {crossing_count} quantile crossing rows.")

    return df, orig_col, dest_col, year_col


df, orig_col, dest_col, year_col = load_supervised_table(
    paths.supervised_path,
    manifest_feature_columns,
    config.label_columns,
)

print("Data shape        :", df.shape)
print("Origin column     :", orig_col)
print("Destination column:", dest_col)
print("Year column       :", year_col)
print("Year counts:")
print(df[year_col].value_counts().sort_index())

Data shape        : (73972, 92)
Origin column     : faf_orig
Destination column: faf_dest
Year column       : year
Year counts:
year
2018     9936
2019    10704
2020    10761
2021    10721
2022    10651
2023    10625
2024    10574
Name: count, dtype: int64


## 6. Temporal split and sample weights

This cell creates the train, validation, and test masks.

The sample weight follows the same principle as the earlier experiments:

\[
w_e = \log(1 + \texttt{obs\_weight\_sum}_e)
\]

The weights are normalized by the train-set mean and clipped to a stable range. The weight column is used only for loss weighting and weighted metrics; it is not used as a predictive feature.

In [6]:
# ============================================================
# Cell 6: Temporal split and sample weights
# ============================================================


def make_temporal_masks(df: pd.DataFrame, year_col: str, cfg: ExperimentConfig) -> Dict[str, np.ndarray]:
    """Create boolean masks for train, validation, and test splits."""
    years = df[year_col].to_numpy()
    masks = {
        "train": np.isin(years, np.array(cfg.train_years, dtype=int)),
        "val": years == int(cfg.val_year),
        "test": years == int(cfg.test_year),
    }
    for split, mask in masks.items():
        if int(mask.sum()) == 0:
            raise ValueError(f"The {split} split is empty. Check the year column and split configuration.")
    return masks


def make_sample_weights(df: pd.DataFrame, train_mask: np.ndarray, cfg: ExperimentConfig) -> np.ndarray:
    """Create normalized sample weights using train statistics only."""
    if not cfg.use_sample_weight or cfg.weight_column not in df.columns:
        return np.ones(len(df), dtype=np.float32)

    raw = pd.to_numeric(df[cfg.weight_column], errors="coerce").to_numpy(dtype=np.float64)
    raw = np.nan_to_num(raw, nan=0.0, posinf=0.0, neginf=0.0)
    raw = np.maximum(raw, 0.0)
    weights = np.log1p(raw)

    train_mean = float(np.mean(weights[train_mask]))
    if not np.isfinite(train_mean) or train_mean <= 0:
        return np.ones(len(df), dtype=np.float32)

    weights = weights / train_mean
    weights = np.clip(weights, cfg.weight_clip_min, cfg.weight_clip_max)
    return weights.astype(np.float32)


masks = make_temporal_masks(df, year_col, config)
sample_weights = make_sample_weights(df, masks["train"], config)

for split_name, split_mask in masks.items():
    print(f"{split_name:>5} rows:", int(split_mask.sum()))

print("Sample weight summary:")
print(pd.Series(sample_weights).describe())

train rows: 52773
  val rows: 10625
 test rows: 10574
Sample weight summary:
count    73972.000000
mean         0.994706
std          0.335815
min          0.124673
25%          0.756137
50%          0.996399
75%          1.236353
max          2.099925
dtype: float64


## 7. Train-only OD historical prior features

This cell constructs the historical prior used by both the reference baseline and the residual MLP models.

For validation and test rows:

```text
historical prior = median target for the same FAF OD pair over train years 2018--2022
```

For train rows:

```text
historical prior = leave-one-training-row-out median for the same FAF OD pair
```

If a row has no available same-OD historical prior, the fallback is the train-set global median.

This design prevents target leakage while preserving the strong temporal-persistence signal discovered in M0 and M0.5.

In [7]:
# ============================================================
# Cell 7: Historical prior construction
# ============================================================


def build_historical_prior_features(
    df: pd.DataFrame,
    masks: Mapping[str, np.ndarray],
    orig_col: str,
    dest_col: str,
    label_columns: Sequence[str],
) -> Tuple[pd.DataFrame, Dict[str, float]]:
    """Construct train-only historical prior features for each OD-year row.

    Train rows use leave-one-training-row-out priors. Validation and test rows
    use all available training years for the same OD pair. Cold OD pairs fall
    back to global train medians.
    """
    n = len(df)
    label_columns = list(label_columns)
    train_mask = np.asarray(masks["train"], dtype=bool)

    global_medians_series = df.loc[train_mask, label_columns].median()
    global_medians = {col: float(global_medians_series[col]) for col in label_columns}
    global_arr = global_medians_series.to_numpy(dtype=np.float64)

    # Store train label arrays by OD pair with original positional indices.
    train_pos = np.flatnonzero(train_mask)
    train_subset = df.loc[train_mask, [orig_col, dest_col] + label_columns].copy()
    train_subset["__pos"] = train_pos

    group_cache: Dict[Tuple[Any, Any], Tuple[np.ndarray, np.ndarray]] = {}
    for key, group in train_subset.groupby([orig_col, dest_col], sort=False):
        positions = group["__pos"].to_numpy(dtype=np.int64)
        values = group[label_columns].to_numpy(dtype=np.float64)
        group_cache[key] = (positions, values)

    prior = np.zeros((n, len(label_columns)), dtype=np.float64)
    n_prior_years = np.zeros(n, dtype=np.float64)
    has_prior = np.zeros(n, dtype=np.float64)

    origins = df[orig_col].to_numpy()
    destinations = df[dest_col].to_numpy()

    for i, key in enumerate(zip(origins, destinations)):
        cached = group_cache.get(key)
        if cached is None:
            prior[i, :] = global_arr
            continue

        positions, values = cached
        if train_mask[i]:
            keep = positions != i
            values_for_prior = values[keep]
        else:
            values_for_prior = values

        if values_for_prior.shape[0] == 0:
            prior[i, :] = global_arr
            continue

        prior[i, :] = np.median(values_for_prior, axis=0)
        n_prior_years[i] = float(values_for_prior.shape[0])
        has_prior[i] = 1.0

    prior_df = pd.DataFrame(
        {
            "hist_truck_q25": prior[:, 0],
            "hist_truck_q50": prior[:, 1],
            "hist_truck_q75": prior[:, 2],
            "hist_truck_iqr": prior[:, 2] - prior[:, 0],
            "hist_truck_q75_q50_gap": prior[:, 2] - prior[:, 1],
            "hist_truck_q50_q25_gap": prior[:, 1] - prior[:, 0],
            "hist_n_prior_years": n_prior_years,
            "hist_has_prior": has_prior,
        },
        index=df.index,
    )

    return prior_df, global_medians


historical_prior_df, global_train_medians = build_historical_prior_features(
    df=df,
    masks=masks,
    orig_col=orig_col,
    dest_col=dest_col,
    label_columns=config.label_columns,
)

for col in HISTORICAL_PRIOR_FEATURE_COLUMNS:
    df[col] = historical_prior_df[col].astype(np.float32)

print("Global train medians:", global_train_medians)
print("Historical prior feature summary:")
print(df[HISTORICAL_PRIOR_FEATURE_COLUMNS].describe().T[["mean", "std", "min", "50%", "max"]])

Global train medians: {'truck_q25': 1494.1099999999997, 'truck_q50': 2314.45, 'truck_q75': 3631.97}
Historical prior feature summary:
                               mean          std   min          50%  \
hist_truck_q25          1534.076416   842.795410  1.00  1494.109985   
hist_truck_q50          2295.498291  1124.945923  1.00  2320.489990   
hist_truck_q75          3731.151123  1716.337769  5.00  3653.560059   
hist_truck_iqr          2197.074463   985.372986  4.00  2079.994995   
hist_truck_q75_q50_gap  1435.652466   675.226379  3.37  1373.587463   
hist_truck_q50_q25_gap   761.421936   362.419098  0.00   730.750000   
hist_n_prior_years         4.220949     0.531482  0.00     4.000000   
hist_has_prior             0.999635     0.019102  0.00     1.000000   

                                 max  
hist_truck_q25           6438.669922  
hist_truck_q50           8321.299805  
hist_truck_q75          12506.919922  
hist_truck_iqr           7540.899902  
hist_truck_q75_q50_gap   5378.9

## 8. Numeric preprocessing

The notebook does not use scikit-learn. This avoids unnecessary dependency issues and keeps preprocessing transparent.

For each feature set, the preprocessor is fitted using only train rows:

1. Convert values to numeric.
2. Replace infinite values with missing values.
3. Impute missing values with the train median.
4. Standardize using train mean and train standard deviation.

Separate preprocessors are fitted for:

- current manifest features only
- current manifest features plus historical prior features

In [8]:
# ============================================================
# Cell 8: Numeric preprocessing utilities
# ============================================================

@dataclass
class NumericFeaturePreprocessor:
    """Train-only median imputation and z-score scaling for numeric features."""

    columns: List[str]
    medians: Optional[np.ndarray] = None
    means: Optional[np.ndarray] = None
    stds: Optional[np.ndarray] = None

    @staticmethod
    def _to_numeric_matrix(df: pd.DataFrame, columns: Sequence[str]) -> np.ndarray:
        """Convert selected columns to a finite numeric matrix with NaNs preserved."""
        numeric = pd.DataFrame(index=df.index)
        for col in columns:
            numeric[col] = pd.to_numeric(df[col], errors="coerce")
        arr = numeric.to_numpy(dtype=np.float64)
        arr[~np.isfinite(arr)] = np.nan
        return arr

    def fit(self, df: pd.DataFrame, train_mask: np.ndarray) -> "NumericFeaturePreprocessor":
        """Fit imputation and scaling statistics on train rows only."""
        arr_train = self._to_numeric_matrix(df.loc[train_mask], self.columns)
        medians = np.nanmedian(arr_train, axis=0)
        medians = np.where(np.isfinite(medians), medians, 0.0)

        filled = np.where(np.isnan(arr_train), medians[None, :], arr_train)
        means = np.mean(filled, axis=0)
        stds = np.std(filled, axis=0)
        stds = np.where((stds < 1e-8) | (~np.isfinite(stds)), 1.0, stds)

        self.medians = medians.astype(np.float64)
        self.means = means.astype(np.float64)
        self.stds = stds.astype(np.float64)
        return self

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        """Transform rows using previously fitted train statistics."""
        if self.medians is None or self.means is None or self.stds is None:
            raise RuntimeError("The preprocessor must be fitted before calling transform().")
        arr = self._to_numeric_matrix(df, self.columns)
        arr = np.where(np.isnan(arr), self.medians[None, :], arr)
        arr = (arr - self.means[None, :]) / self.stds[None, :]
        arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
        return arr.astype(np.float32)

    def to_dict(self) -> Dict[str, Any]:
        """Serialize preprocessing statistics for reproducibility."""
        return {
            "columns": self.columns,
            "medians": None if self.medians is None else self.medians.tolist(),
            "means": None if self.means is None else self.means.tolist(),
            "stds": None if self.stds is None else self.stds.tolist(),
        }


current_preprocessor = NumericFeaturePreprocessor(columns=list(manifest_feature_columns)).fit(df, masks["train"])
prior_preprocessor = NumericFeaturePreprocessor(columns=list(feature_columns_with_prior)).fit(df, masks["train"])

X_current_all = current_preprocessor.transform(df)
X_prior_all = prior_preprocessor.transform(df)

print("X_current_all shape:", X_current_all.shape)
print("X_prior_all shape  :", X_prior_all.shape)

X_current_all shape: (73972, 64)
X_prior_all shape  : (73972, 72)


## 9. Target scaling and tensor dataset construction

The MLP is trained on scaled target values for numerical stability. The default scale is the train-set median of `truck_q50`.

All reported metrics are converted back to original minutes.

The residual models receive both:

- scaled targets
- scaled historical priors

The historical priors are used inside the residualized monotone head.

In [9]:
# ============================================================
# Cell 9: Target scaling and tensor dataset utilities
# ============================================================

class SupervisedTensorDataset(Dataset):
    """A compact PyTorch dataset for supervised OD-year rows."""

    def __init__(self, X: np.ndarray, y: np.ndarray, prior: np.ndarray, weight: np.ndarray):
        self.X = torch.as_tensor(X, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.float32)
        self.prior = torch.as_tensor(prior, dtype=torch.float32)
        self.weight = torch.as_tensor(weight, dtype=torch.float32)

    def __len__(self) -> int:
        return int(self.X.shape[0])

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        return self.X[idx], self.y[idx], self.prior[idx], self.weight[idx]


def split_array(arr: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Return rows selected by a boolean mask."""
    return arr[np.asarray(mask, dtype=bool)]


y_all = df[list(config.label_columns)].to_numpy(dtype=np.float32)
prior_all = df[["hist_truck_q25", "hist_truck_q50", "hist_truck_q75"]].to_numpy(dtype=np.float32)

target_scale = float(np.median(y_all[masks["train"], 1]))
if not np.isfinite(target_scale) or target_scale <= 0:
    raise ValueError(f"Invalid target scale: {target_scale}")

y_scaled_all = y_all / target_scale
prior_scaled_all = prior_all / target_scale

global_train_medians_arr = np.array(
    [global_train_medians[col] for col in config.label_columns],
    dtype=np.float32,
)
initial_quantiles_scaled = global_train_medians_arr / target_scale

print("Target scale, minutes:", target_scale)
print("Initial global quantiles, scaled:", initial_quantiles_scaled)

Target scale, minutes: 2314.449951171875
Initial global quantiles, scaled: [0.6455573 1.        1.5692583]


## 10. Metrics

The metrics in this notebook match the earlier baseline analysis and the project evaluation plan.

Core metrics:

- `mae_q25`, `mae_q50`, `mae_q75`
- `rmse_q50`
- `pinball_q25`, `pinball_q50`, `pinball_q75`, `pinball_mean`
- `weighted_pinball_mean`
- `iqr_mae`
- `stress_top10_mae_q75`
- `stress_top10_iqr_mae`
- `sparse_bottom25_mae_q75`
- `raw_crossing_rate`

The stress subset is defined within each split by the top 10% of true `truck_q75`. The sparse subset uses the bottom 25% of `n_fmi_county_pairs` when that diagnostic column is available.

In [10]:
# ============================================================
# Cell 10: Metric utilities
# ============================================================


def postprocess_quantiles(pred: np.ndarray) -> np.ndarray:
    """Clip predictions to nonnegative values and enforce monotone quantiles by sorting."""
    pred = np.asarray(pred, dtype=np.float64)
    pred = np.nan_to_num(pred, nan=0.0, posinf=0.0, neginf=0.0)
    pred = np.maximum(pred, 0.0)
    pred = np.sort(pred, axis=1)
    return pred


def raw_crossing_rate(pred: np.ndarray) -> float:
    """Compute the fraction of rows with q25 > q50 or q50 > q75 before postprocessing."""
    pred = np.asarray(pred, dtype=np.float64)
    crossing = (pred[:, 0] > pred[:, 1]) | (pred[:, 1] > pred[:, 2])
    return float(np.mean(crossing))


def pinball_loss_np(y_true: np.ndarray, y_pred: np.ndarray, taus: Sequence[float]) -> np.ndarray:
    """Return per-row, per-quantile pinball losses in original units."""
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    taus_arr = np.asarray(taus, dtype=np.float64)[None, :]
    diff = y_true - y_pred
    return np.maximum(taus_arr * diff, (taus_arr - 1.0) * diff)


def weighted_average(values: np.ndarray, weights: Optional[np.ndarray]) -> float:
    """Compute a stable weighted average for one-dimensional values."""
    values = np.asarray(values, dtype=np.float64)
    if weights is None:
        return float(np.mean(values))
    weights = np.asarray(weights, dtype=np.float64)
    denom = float(np.sum(weights))
    if denom <= 0 or not np.isfinite(denom):
        return float(np.mean(values))
    return float(np.sum(values * weights) / denom)


def evaluate_predictions(
    y_true: np.ndarray,
    pred_raw: np.ndarray,
    weights: np.ndarray,
    df_split: pd.DataFrame,
    cfg: ExperimentConfig,
) -> Dict[str, float]:
    """Compute all evaluation metrics for a split."""
    y_true = np.asarray(y_true, dtype=np.float64)
    pred_raw = np.asarray(pred_raw, dtype=np.float64)
    pred = postprocess_quantiles(pred_raw)
    weights = np.asarray(weights, dtype=np.float64)

    pinball = pinball_loss_np(y_true, pred, cfg.taus)
    row_pinball_mean = np.mean(pinball, axis=1)

    true_iqr = y_true[:, 2] - y_true[:, 0]
    pred_iqr = pred[:, 2] - pred[:, 0]

    metrics: Dict[str, float] = {
        "n_eval": float(len(y_true)),
        "mae_q25": float(np.mean(np.abs(pred[:, 0] - y_true[:, 0]))),
        "mae_q50": float(np.mean(np.abs(pred[:, 1] - y_true[:, 1]))),
        "mae_q75": float(np.mean(np.abs(pred[:, 2] - y_true[:, 2]))),
        "rmse_q50": float(np.sqrt(np.mean((pred[:, 1] - y_true[:, 1]) ** 2))),
        "pinball_q25": float(np.mean(pinball[:, 0])),
        "pinball_q50": float(np.mean(pinball[:, 1])),
        "pinball_q75": float(np.mean(pinball[:, 2])),
        "pinball_mean": float(np.mean(row_pinball_mean)),
        "weighted_pinball_mean": weighted_average(row_pinball_mean, weights),
        "iqr_mae": float(np.mean(np.abs(pred_iqr - true_iqr))),
        "q50_inside_pred_q25_q75_rate": float(np.mean((y_true[:, 1] >= pred[:, 0]) & (y_true[:, 1] <= pred[:, 2]))),
        "bias_q25": float(np.mean(pred[:, 0] - y_true[:, 0])),
        "bias_q50": float(np.mean(pred[:, 1] - y_true[:, 1])),
        "bias_q75": float(np.mean(pred[:, 2] - y_true[:, 2])),
        "pred_iqr_mean": float(np.mean(pred_iqr)),
        "true_iqr_mean": float(np.mean(true_iqr)),
        "raw_crossing_rate": raw_crossing_rate(pred_raw),
    }

    # Stress subset: top 10% true q75 within the current split.
    q75_threshold = float(np.quantile(y_true[:, 2], 0.90))
    stress_mask = y_true[:, 2] >= q75_threshold
    metrics["stress_top10_threshold_q75"] = q75_threshold
    metrics["stress_top10_n"] = float(np.sum(stress_mask))
    metrics["stress_top10_mae_q75"] = float(np.mean(np.abs(pred[stress_mask, 2] - y_true[stress_mask, 2])))
    metrics["stress_top10_iqr_mae"] = float(np.mean(np.abs(pred_iqr[stress_mask] - true_iqr[stress_mask])))

    # Top-IQR subset: useful for distributional-risk diagnostics.
    iqr_threshold = float(np.quantile(true_iqr, 0.90))
    top_iqr_mask = true_iqr >= iqr_threshold
    metrics["stress_top10_threshold_iqr"] = iqr_threshold
    metrics["stress_top10_n_iqr"] = float(np.sum(top_iqr_mask))
    metrics["stress_top10_mae_iqr_subset_q75"] = float(np.mean(np.abs(pred[top_iqr_mask, 2] - y_true[top_iqr_mask, 2])))
    metrics["stress_top10_iqr_subset_iqr_mae"] = float(np.mean(np.abs(pred_iqr[top_iqr_mask] - true_iqr[top_iqr_mask])))

    # Sparse-label subset: bottom 25% n_fmi_county_pairs if available.
    if "n_fmi_county_pairs" in df_split.columns:
        support = pd.to_numeric(df_split["n_fmi_county_pairs"], errors="coerce").to_numpy(dtype=np.float64)
        support = np.nan_to_num(support, nan=np.nanmedian(support))
        sparse_threshold = float(np.quantile(support, 0.25))
        sparse_mask = support <= sparse_threshold
        metrics["sparse_bottom25_threshold_n_fmi_county_pairs"] = sparse_threshold
        metrics["sparse_bottom25_n"] = float(np.sum(sparse_mask))
        metrics["sparse_bottom25_mae_q75"] = float(np.mean(np.abs(pred[sparse_mask, 2] - y_true[sparse_mask, 2])))
        metrics["sparse_bottom25_iqr_mae"] = float(np.mean(np.abs(pred_iqr[sparse_mask] - true_iqr[sparse_mask])))
    else:
        metrics["sparse_bottom25_threshold_n_fmi_county_pairs"] = float("nan")
        metrics["sparse_bottom25_n"] = float("nan")
        metrics["sparse_bottom25_mae_q75"] = float("nan")
        metrics["sparse_bottom25_iqr_mae"] = float("nan")

    return metrics

## 11. PyTorch model definitions

This cell defines two MLP model families.

### DirectMonotoneMLP

This model predicts quantiles directly. The output head guarantees monotonicity by predicting a positive q25 and positive quantile gaps.

### ResidualMonotoneMLP

This model is initialized exactly around the historical prior. It predicts residual corrections to:

- q25 location
- q50 - q25 gap
- q75 - q50 gap

The residual head also guarantees monotonicity by construction. This is the main model for this experiment.

In [11]:
# ============================================================
# Cell 11: PyTorch MLP model definitions
# ============================================================


def inverse_softplus_tensor(x: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """Numerically stable inverse of softplus for positive tensors."""
    x = torch.clamp(x, min=eps)
    return torch.where(x > 20.0, x, torch.log(torch.expm1(x)))


class FeedForwardBackbone(nn.Module):
    """A simple normalized MLP backbone for tabular features."""

    def __init__(self, input_dim: int, hidden_dim: int, n_hidden_layers: int, dropout: float):
        super().__init__()
        if n_hidden_layers < 1:
            raise ValueError("n_hidden_layers must be at least 1.")

        layers: List[nn.Module] = []
        in_dim = input_dim
        for _ in range(n_hidden_layers):
            layers.append(nn.Linear(in_dim, hidden_dim))
            layers.append(nn.LayerNorm(hidden_dim))
            layers.append(nn.SiLU())
            layers.append(nn.Dropout(dropout))
            in_dim = hidden_dim
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class DirectMonotoneMLP(nn.Module):
    """Direct MLP quantile model with a cumulative softplus monotone head."""

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        n_hidden_layers: int,
        dropout: float,
        initial_quantiles_scaled: np.ndarray,
        eps: float = 1e-6,
    ):
        super().__init__()
        self.backbone = FeedForwardBackbone(input_dim, hidden_dim, n_hidden_layers, dropout)
        self.out = nn.Linear(hidden_dim, 3)
        self.eps = eps
        self._initialize_head(initial_quantiles_scaled)

    def _initialize_head(self, initial_quantiles_scaled: np.ndarray) -> None:
        """Initialize the direct head to predict global train quantiles before learning."""
        q25, q50, q75 = [float(x) for x in initial_quantiles_scaled]
        init_gaps = torch.tensor(
            [q25, max(q50 - q25, self.eps), max(q75 - q50, self.eps)],
            dtype=torch.float32,
        )
        with torch.no_grad():
            self.out.weight.zero_()
            self.out.bias.copy_(inverse_softplus_tensor(init_gaps, eps=self.eps))

    def forward(self, x: torch.Tensor, prior_scaled: Optional[torch.Tensor] = None) -> torch.Tensor:
        hidden = self.backbone(x)
        raw = self.out(hidden)
        q25 = F.softplus(raw[:, 0])
        gap50 = F.softplus(raw[:, 1])
        gap75 = F.softplus(raw[:, 2])
        q50 = q25 + gap50
        q75 = q50 + gap75
        return torch.stack([q25, q50, q75], dim=1)


class ResidualMonotoneMLP(nn.Module):
    """Residual MLP initialized around a historical-prior monotone quantile vector."""

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        n_hidden_layers: int,
        dropout: float,
        eps: float = 1e-6,
    ):
        super().__init__()
        self.backbone = FeedForwardBackbone(input_dim, hidden_dim, n_hidden_layers, dropout)
        self.out = nn.Linear(hidden_dim, 3)
        self.eps = eps
        self._initialize_zero_residual_head()

    def _initialize_zero_residual_head(self) -> None:
        """Initialize the residual head so the model starts exactly at the historical prior."""
        with torch.no_grad():
            self.out.weight.zero_()
            self.out.bias.zero_()

    def forward(self, x: torch.Tensor, prior_scaled: torch.Tensor) -> torch.Tensor:
        if prior_scaled is None:
            raise ValueError("ResidualMonotoneMLP requires prior_scaled in forward().")

        hidden = self.backbone(x)
        delta = self.out(hidden)

        base_q25 = torch.clamp(prior_scaled[:, 0], min=self.eps)
        base_gap50 = torch.clamp(prior_scaled[:, 1] - prior_scaled[:, 0], min=self.eps)
        base_gap75 = torch.clamp(prior_scaled[:, 2] - prior_scaled[:, 1], min=self.eps)

        q25 = F.softplus(inverse_softplus_tensor(base_q25, eps=self.eps) + delta[:, 0])
        gap50 = F.softplus(inverse_softplus_tensor(base_gap50, eps=self.eps) + delta[:, 1])
        gap75 = F.softplus(inverse_softplus_tensor(base_gap75, eps=self.eps) + delta[:, 2])
        q50 = q25 + gap50
        q75 = q50 + gap75
        return torch.stack([q25, q50, q75], dim=1)

## 12. Training utilities

This cell defines the training loop, validation logic, and checkpointing.

Important implementation details:

- The loss is computed in scaled minutes for numerical stability.
- Early stopping monitors validation weighted pinball loss in original minutes.
- The best validation checkpoint is restored before test evaluation.
- Gradient clipping is used to stabilize training.

In [12]:
# ============================================================
# Cell 12: Training utilities
# ============================================================


def pinball_loss_torch(pred: torch.Tensor, target: torch.Tensor, taus: Sequence[float]) -> torch.Tensor:
    """Return per-row, per-quantile pinball losses."""
    tau_tensor = torch.as_tensor(taus, dtype=pred.dtype, device=pred.device).view(1, -1)
    diff = target - pred
    return torch.maximum(tau_tensor * diff, (tau_tensor - 1.0) * diff)


def weighted_torch_mean(values: torch.Tensor, weights: torch.Tensor) -> torch.Tensor:
    """Compute a stable weighted mean for a batch."""
    denom = torch.clamp(weights.sum(), min=1e-8)
    return (values * weights).sum() / denom


def make_loader(
    X: np.ndarray,
    y_scaled: np.ndarray,
    prior_scaled: np.ndarray,
    weights: np.ndarray,
    batch_size: int,
    shuffle: bool,
    cfg: ExperimentConfig,
    device: torch.device,
) -> DataLoader:
    """Create a PyTorch DataLoader for one split."""
    dataset = SupervisedTensorDataset(X, y_scaled, prior_scaled, weights)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        drop_last=False,
    )


@torch.no_grad()
def predict_scaled(
    model: nn.Module,
    X: np.ndarray,
    y_scaled: np.ndarray,
    prior_scaled: np.ndarray,
    weights: np.ndarray,
    cfg: ExperimentConfig,
    device: torch.device,
) -> np.ndarray:
    """Predict scaled quantiles for a full split."""
    model.eval()
    loader = make_loader(
        X=X,
        y_scaled=y_scaled,
        prior_scaled=prior_scaled,
        weights=weights,
        batch_size=cfg.batch_size,
        shuffle=False,
        cfg=cfg,
        device=device,
    )
    preds: List[np.ndarray] = []
    for xb, _, priorb, _ in loader:
        xb = xb.to(device, non_blocking=True)
        priorb = priorb.to(device, non_blocking=True)
        pred = model(xb, priorb)
        preds.append(pred.detach().cpu().numpy())
    return np.vstack(preds).astype(np.float32)


def train_one_mlp_model(
    model_name: str,
    model: nn.Module,
    X_all: np.ndarray,
    y_scaled_all: np.ndarray,
    prior_scaled_all: np.ndarray,
    weights_all: np.ndarray,
    masks: Mapping[str, np.ndarray],
    df: pd.DataFrame,
    cfg: ExperimentConfig,
    target_scale: float,
    device: torch.device,
    model_path: Optional[Path] = None,
) -> Tuple[nn.Module, pd.DataFrame, Dict[str, np.ndarray]]:
    """Train a single MLP model and return the best model, history, and split predictions."""
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)

    X_train = split_array(X_all, masks["train"])
    y_train = split_array(y_scaled_all, masks["train"])
    prior_train = split_array(prior_scaled_all, masks["train"])
    w_train = split_array(weights_all, masks["train"])

    train_loader = make_loader(
        X=X_train,
        y_scaled=y_train,
        prior_scaled=prior_train,
        weights=w_train,
        batch_size=cfg.batch_size,
        shuffle=True,
        cfg=cfg,
        device=device,
    )

    best_metric = float("inf")
    best_state: Optional[Dict[str, torch.Tensor]] = None
    best_epoch = -1
    epochs_without_improvement = 0
    history_rows: List[Dict[str, float]] = []

    for epoch in range(1, cfg.max_epochs + 1):
        model.train()
        train_losses: List[float] = []

        for xb, yb, priorb, wb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            priorb = priorb.to(device, non_blocking=True)
            wb = wb.to(device, non_blocking=True)

            pred = model(xb, priorb)
            per_row_pinball = pinball_loss_torch(pred, yb, cfg.taus).mean(dim=1)
            loss = weighted_torch_mean(per_row_pinball, wb)

            if cfg.lambda_iqr > 0:
                pred_iqr = pred[:, 2] - pred[:, 0]
                true_iqr = yb[:, 2] - yb[:, 0]
                iqr_loss = F.smooth_l1_loss(pred_iqr, true_iqr, reduction="none")
                loss = loss + cfg.lambda_iqr * weighted_torch_mean(iqr_loss, wb)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
            optimizer.step()

            train_losses.append(float(loss.detach().cpu().item()))

        # Validation in original minutes.
        val_pred_scaled = predict_scaled(
            model=model,
            X=split_array(X_all, masks["val"]),
            y_scaled=split_array(y_scaled_all, masks["val"]),
            prior_scaled=split_array(prior_scaled_all, masks["val"]),
            weights=split_array(weights_all, masks["val"]),
            cfg=cfg,
            device=device,
        )
        val_pred_minutes = val_pred_scaled * target_scale
        val_metrics = evaluate_predictions(
            y_true=split_array(y_all, masks["val"]),
            pred_raw=val_pred_minutes,
            weights=split_array(weights_all, masks["val"]),
            df_split=df.loc[masks["val"]].copy(),
            cfg=cfg,
        )
        monitor = float(val_metrics["weighted_pinball_mean"])

        history_row = {
            "model": model_name,
            "epoch": float(epoch),
            "train_loss_scaled": float(np.mean(train_losses)),
            "val_weighted_pinball_mean": monitor,
            "val_pinball_mean": float(val_metrics["pinball_mean"]),
            "val_mae_q50": float(val_metrics["mae_q50"]),
            "val_mae_q75": float(val_metrics["mae_q75"]),
            "val_iqr_mae": float(val_metrics["iqr_mae"]),
        }
        history_rows.append(history_row)

        improved = monitor < (best_metric - cfg.min_delta)
        if improved:
            best_metric = monitor
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epoch == 1 or epoch % 10 == 0 or improved:
            print(
                f"[{model_name}] epoch={epoch:03d} "
                f"train_loss={np.mean(train_losses):.6f} "
                f"val_weighted_pinball={monitor:.4f} "
                f"best={best_metric:.4f}@{best_epoch}"
            )

        if epochs_without_improvement >= cfg.patience:
            print(f"[{model_name}] Early stopping at epoch {epoch}. Best epoch: {best_epoch}.")
            break

    if best_state is None:
        raise RuntimeError(f"Training failed to produce a valid checkpoint for {model_name}.")

    model.load_state_dict(best_state)

    if model_path is not None:
        torch.save(
            {
                "model_name": model_name,
                "state_dict": model.state_dict(),
                "best_epoch": best_epoch,
                "best_val_weighted_pinball_mean": best_metric,
                "config": asdict(cfg),
                "target_scale": target_scale,
            },
            model_path,
        )

    split_predictions: Dict[str, np.ndarray] = {}
    for split in ["train", "val", "test"]:
        pred_scaled = predict_scaled(
            model=model,
            X=split_array(X_all, masks[split]),
            y_scaled=split_array(y_scaled_all, masks[split]),
            prior_scaled=split_array(prior_scaled_all, masks[split]),
            weights=split_array(weights_all, masks[split]),
            cfg=cfg,
            device=device,
        )
        split_predictions[split] = pred_scaled * target_scale

    history_df = pd.DataFrame(history_rows)
    return model, history_df, split_predictions

## 13. Reference baselines

Before training neural models, this cell evaluates two reference baselines:

- global train median
- OD historical prior

These baselines are included in the final leaderboard so the MLP models can be compared against the strongest simple historical baseline.

In [13]:
# ============================================================
# Cell 13: Reference baseline predictions
# ============================================================


def repeat_prediction(vector: np.ndarray, n: int) -> np.ndarray:
    """Repeat a single quantile vector for n rows."""
    return np.repeat(vector.reshape(1, -1), repeats=n, axis=0).astype(np.float32)


def make_reference_predictions(
    y_all: np.ndarray,
    prior_all: np.ndarray,
    masks: Mapping[str, np.ndarray],
    global_train_medians_arr: np.ndarray,
) -> Dict[str, Dict[str, np.ndarray]]:
    """Create split-level predictions for non-neural reference baselines."""
    refs: Dict[str, Dict[str, np.ndarray]] = {
        "global_train_median": {},
        "od_historical_prior": {},
    }
    for split, mask in masks.items():
        n_split = int(np.sum(mask))
        refs["global_train_median"][split] = repeat_prediction(global_train_medians_arr, n_split)
        refs["od_historical_prior"][split] = prior_all[mask].astype(np.float32)
    return refs


reference_predictions = make_reference_predictions(
    y_all=y_all,
    prior_all=prior_all,
    masks=masks,
    global_train_medians_arr=global_train_medians_arr,
)

for model_name, split_preds in reference_predictions.items():
    print(model_name, {split: preds.shape for split, preds in split_preds.items()})

global_train_median {'train': (52773, 3), 'val': (10625, 3), 'test': (10574, 3)}
od_historical_prior {'train': (52773, 3), 'val': (10625, 3), 'test': (10574, 3)}


## 14. Train MLP-residual model suite

This cell trains the full MLP suite.

Model definitions:

- `mlp_raw_current_features`: direct MLP using the 64 manifest features.
- `mlp_prior_direct`: direct MLP using the 64 manifest features plus 8 historical-prior features.
- `mlp_residual_current_features`: residual MLP initialized at the historical prior, using 64 manifest features.
- `mlp_residual_prior_features`: residual MLP initialized at the historical prior, using 64 manifest features plus historical-prior features.

The residual models are the main focus of this notebook.

In [14]:
# ============================================================
# Cell 14: Train MLP model suite
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", device)

model_predictions: Dict[str, Dict[str, np.ndarray]] = dict(reference_predictions)
model_histories: List[pd.DataFrame] = []
model_notes: Dict[str, Any] = {
    "global_train_median": {
        "description": "Global train median for q25/q50/q75."
    },
    "od_historical_prior": {
        "description": "Train-only OD historical median. Train rows use leave-one-training-row-out priors.",
        "global_train_medians_for_cold_od": global_train_medians,
    },
}

model_specs: List[Dict[str, Any]] = []
if config.run_mlp_raw_current_features:
    model_specs.append({
        "name": "mlp_raw_current_features",
        "kind": "direct",
        "X": X_current_all,
        "feature_columns": manifest_feature_columns,
        "description": "Direct monotone MLP using only the manifest current OD/zone features.",
    })
if config.run_mlp_prior_direct:
    model_specs.append({
        "name": "mlp_prior_direct",
        "kind": "direct",
        "X": X_prior_all,
        "feature_columns": feature_columns_with_prior,
        "description": "Direct monotone MLP using manifest features plus historical prior features.",
    })
if config.run_mlp_residual_current_features:
    model_specs.append({
        "name": "mlp_residual_current_features",
        "kind": "residual",
        "X": X_current_all,
        "feature_columns": manifest_feature_columns,
        "description": "Residual monotone MLP initialized at the OD historical prior, using current features for correction.",
    })
if config.run_mlp_residual_prior_features:
    model_specs.append({
        "name": "mlp_residual_prior_features",
        "kind": "residual",
        "X": X_prior_all,
        "feature_columns": feature_columns_with_prior,
        "description": "Residual monotone MLP initialized at the OD historical prior, using current and historical prior features for correction.",
    })

for spec in model_specs:
    set_global_seed(config.seed)
    model_name = spec["name"]
    input_dim = int(spec["X"].shape[1])

    if spec["kind"] == "direct":
        model = DirectMonotoneMLP(
            input_dim=input_dim,
            hidden_dim=config.hidden_dim,
            n_hidden_layers=config.n_hidden_layers,
            dropout=config.dropout,
            initial_quantiles_scaled=initial_quantiles_scaled,
            eps=config.positive_epsilon,
        )
    elif spec["kind"] == "residual":
        model = ResidualMonotoneMLP(
            input_dim=input_dim,
            hidden_dim=config.hidden_dim,
            n_hidden_layers=config.n_hidden_layers,
            dropout=config.dropout,
            eps=config.positive_epsilon,
        )
    else:
        raise ValueError(f"Unknown model kind: {spec['kind']}")

    checkpoint_path = paths.models_dir / f"{model_name}.pt" if config.save_models else None
    trained_model, history_df, split_preds = train_one_mlp_model(
        model_name=model_name,
        model=model,
        X_all=spec["X"],
        y_scaled_all=y_scaled_all,
        prior_scaled_all=prior_scaled_all,
        weights_all=sample_weights,
        masks=masks,
        df=df,
        cfg=config,
        target_scale=target_scale,
        device=device,
        model_path=checkpoint_path,
    )

    model_predictions[model_name] = split_preds
    model_histories.append(history_df)
    model_notes[model_name] = {
        "description": spec["description"],
        "kind": spec["kind"],
        "features": list(spec["feature_columns"]),
        "checkpoint_path": str(checkpoint_path) if checkpoint_path is not None else None,
    }

print("Finished training models:", list(model_predictions.keys()))

Training device: cuda


E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Lib\site-packages\torch\nn\modules\module.py:1369: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:40.)
  return t.to(


[mlp_raw_current_features] epoch=001 train_loss=0.175652 val_weighted_pinball=295.1761 best=295.1761@1
[mlp_raw_current_features] epoch=002 train_loss=0.123141 val_weighted_pinball=255.0450 best=255.0450@2
[mlp_raw_current_features] epoch=003 train_loss=0.109693 val_weighted_pinball=235.1904 best=235.1904@3
[mlp_raw_current_features] epoch=004 train_loss=0.104398 val_weighted_pinball=228.1258 best=228.1258@4
[mlp_raw_current_features] epoch=005 train_loss=0.101341 val_weighted_pinball=222.2209 best=222.2209@5
[mlp_raw_current_features] epoch=006 train_loss=0.099537 val_weighted_pinball=220.1133 best=220.1133@6
[mlp_raw_current_features] epoch=007 train_loss=0.098317 val_weighted_pinball=218.6862 best=218.6862@7
[mlp_raw_current_features] epoch=008 train_loss=0.096713 val_weighted_pinball=215.8002 best=215.8002@8
[mlp_raw_current_features] epoch=009 train_loss=0.095636 val_weighted_pinball=213.8851 best=213.8851@9
[mlp_raw_current_features] epoch=010 train_loss=0.094115 val_weighted_pin

## 15. Optional validation-year residual calibration

Global validation residual calibration is disabled by default because the earlier M0 results showed that simple global calibration may not transfer well from 2023 to 2024.

The function is included for reproducibility and ablation, but the main MLP result should usually use the uncalibrated monotone model output.

In [15]:
# ============================================================
# Cell 15: Optional global validation residual calibration
# ============================================================


def apply_global_validation_median_calibration(
    model_predictions: Dict[str, Dict[str, np.ndarray]],
    y_all: np.ndarray,
    masks: Mapping[str, np.ndarray],
) -> Dict[str, Dict[str, np.ndarray]]:
    """Add calibrated prediction variants using validation median residuals."""
    augmented = dict(model_predictions)
    y_val = y_all[masks["val"]]

    for model_name, split_preds in list(model_predictions.items()):
        if model_name.endswith("_calibrated"):
            continue
        val_pred = postprocess_quantiles(split_preds["val"])
        correction = np.median(y_val - val_pred, axis=0)
        calibrated_name = f"{model_name}_global_val_median_calibrated"
        augmented[calibrated_name] = {}
        for split, pred in split_preds.items():
            augmented[calibrated_name][split] = pred + correction.reshape(1, -1)
        model_notes[calibrated_name] = {
            "description": "Global validation-year median residual calibration applied to model predictions.",
            "base_model": model_name,
            "correction_q25_q50_q75": correction.tolist(),
        }
    return augmented


if config.run_global_val_median_calibration:
    model_predictions = apply_global_validation_median_calibration(model_predictions, y_all, masks)
    print("Calibration variants added.")
else:
    print("Global validation median calibration is disabled for the main MLP-residual run.")

Global validation median calibration is disabled for the main MLP-residual run.


## 16. Evaluate all models

This cell evaluates all reference and MLP models on train, validation, and test splits.

The test leaderboard is sorted by `pinball_mean`, while `weighted_pinball_mean` is also reported because the training objective uses observation-support weights.

In [16]:
# ============================================================
# Cell 16: Evaluation and leaderboard construction
# ============================================================

metrics_rows: List[Dict[str, Any]] = []

for model_name, split_preds in model_predictions.items():
    for split, pred_raw in split_preds.items():
        split_mask = masks[split]
        y_split = y_all[split_mask]
        w_split = sample_weights[split_mask]
        df_split = df.loc[split_mask].copy()
        metrics = evaluate_predictions(
            y_true=y_split,
            pred_raw=pred_raw,
            weights=w_split,
            df_split=df_split,
            cfg=config,
        )
        row = {
            "model": model_name,
            "variant": "postprocessed",
            "split": split,
        }
        row.update(metrics)
        metrics_rows.append(row)

metrics_df = pd.DataFrame(metrics_rows)
leaderboard_test = metrics_df.loc[metrics_df["split"] == "test"].sort_values(
    by=["pinball_mean", "mae_q75"], ascending=[True, True]
).reset_index(drop=True)
leaderboard_test.insert(0, "rank", np.arange(1, len(leaderboard_test) + 1))

print("Test leaderboard:")
display_columns = [
    "rank", "model", "pinball_mean", "weighted_pinball_mean", "mae_q50", "mae_q75",
    "iqr_mae", "stress_top10_mae_q75", "sparse_bottom25_mae_q75", "raw_crossing_rate",
]
print(leaderboard_test[display_columns].to_string(index=False))

Test leaderboard:
 rank                         model  pinball_mean  weighted_pinball_mean    mae_q50     mae_q75    iqr_mae  stress_top10_mae_q75  sparse_bottom25_mae_q75  raw_crossing_rate
    1 mlp_residual_current_features    135.933607             120.340749 242.287672  455.564742 376.481644            641.957466               565.448160                0.0
    2   mlp_residual_prior_features    138.905682             123.590372 239.137838  458.999342 375.083455            623.819621               561.111913                0.0
    3      mlp_raw_current_features    149.077044             131.035618 273.724012  501.053885 441.746506            687.172521               656.870316                0.0
    4           od_historical_prior    153.000149             136.248358 230.534636  472.672179 385.553897            709.868525               577.776243                0.0
    5              mlp_prior_direct    167.614298             154.271500 292.544509  516.692091 388.420538           

## 17. Build prediction output table

This cell creates a row-level prediction table for validation and test years.

The output includes:

- OD identifiers
- year and split
- true labels
- predicted quantiles
- historical prior quantiles
- signed errors and absolute errors
- diagnostic support columns when available

The row-level prediction table is necessary for later error analysis by q75 decile, IQR decile, sparse-label bucket, origin zone, destination zone, and model disagreement.

In [17]:
# ============================================================
# Cell 17: Prediction output table construction
# ============================================================


def build_prediction_rows_for_split(
    model_name: str,
    split: str,
    pred_raw: np.ndarray,
    df: pd.DataFrame,
    masks: Mapping[str, np.ndarray],
    label_columns: Sequence[str],
    orig_col: str,
    dest_col: str,
    year_col: str,
    sample_weights: np.ndarray,
) -> pd.DataFrame:
    """Create a prediction DataFrame for one model and one split."""
    split_mask = masks[split]
    base = df.loc[split_mask, [orig_col, dest_col, year_col] + list(label_columns)].copy().reset_index(drop=True)
    pred = postprocess_quantiles(pred_raw)
    prior = prior_all[split_mask]

    out = pd.DataFrame({
        "model": model_name,
        "variant": "postprocessed",
        "split": split,
        orig_col: base[orig_col].to_numpy(),
        dest_col: base[dest_col].to_numpy(),
        year_col: base[year_col].to_numpy(),
        "true_q25": base[label_columns[0]].to_numpy(dtype=np.float64),
        "true_q50": base[label_columns[1]].to_numpy(dtype=np.float64),
        "true_q75": base[label_columns[2]].to_numpy(dtype=np.float64),
        "pred_q25": pred[:, 0],
        "pred_q50": pred[:, 1],
        "pred_q75": pred[:, 2],
        "hist_prior_q25": prior[:, 0],
        "hist_prior_q50": prior[:, 1],
        "hist_prior_q75": prior[:, 2],
        "sample_weight": sample_weights[split_mask],
    })

    out["error_q25"] = out["pred_q25"] - out["true_q25"]
    out["error_q50"] = out["pred_q50"] - out["true_q50"]
    out["error_q75"] = out["pred_q75"] - out["true_q75"]
    out["abs_error_q25"] = np.abs(out["error_q25"])
    out["abs_error_q50"] = np.abs(out["error_q50"])
    out["abs_error_q75"] = np.abs(out["error_q75"])
    out["true_iqr"] = out["true_q75"] - out["true_q25"]
    out["pred_iqr"] = out["pred_q75"] - out["pred_q25"]
    out["abs_error_iqr"] = np.abs(out["pred_iqr"] - out["true_iqr"])

    # Add diagnostics when available. These are not predictive features.
    for diag_col in ["n_fmi_county_pairs", "obs_weight_sum"]:
        if diag_col in df.columns:
            out[diag_col] = df.loc[split_mask, diag_col].to_numpy()

    return out


prediction_frames: List[pd.DataFrame] = []
for model_name, split_preds in model_predictions.items():
    for split in ["val", "test"]:
        prediction_frames.append(
            build_prediction_rows_for_split(
                model_name=model_name,
                split=split,
                pred_raw=split_preds[split],
                df=df,
                masks=masks,
                label_columns=config.label_columns,
                orig_col=orig_col,
                dest_col=dest_col,
                year_col=year_col,
                sample_weights=sample_weights,
            )
        )

predictions_df = pd.concat(prediction_frames, ignore_index=True)
print("Predictions table shape:", predictions_df.shape)
print(predictions_df.head())

Predictions table shape: (127194, 27)
                 model        variant split faf_orig faf_dest  year  true_q25  \
0  global_train_median  postprocessed   val      011      011  2023     24.04   
1  global_train_median  postprocessed   val      011      012  2023    225.00   
2  global_train_median  postprocessed   val      011      019  2023     75.10   
3  global_train_median  postprocessed   val      011      050  2023    486.83   
4  global_train_median  postprocessed   val      011      091  2023   1920.00   

   true_q50  true_q75     pred_q25  ...    error_q50    error_q75  \
0     60.00    137.73  1494.109985  ...  2254.449951  3494.239971   
1    675.00   1563.57  1494.109985  ...  1639.449951  2068.399971   
2    164.98    576.03  1494.109985  ...  2149.469951  3055.939971   
3   1166.63   1764.33  1494.109985  ...  1147.819951  1867.639971   
4   2783.55   3918.35  1494.109985  ...  -469.100049  -286.380029   

   abs_error_q25  abs_error_q50  abs_error_q75  true_iqr    

## 18. Save experiment artifacts

This cell writes all output artifacts to disk.

Files saved:

```text
metrics.csv
leaderboard_test.csv
predictions_val_test_mlp_residual.parquet
predictions_val_test.parquet
training_history.csv
feature_columns_used.json
model_notes.json
run_config.json
models/*.pt
```

The explicit prediction filename prevents confusion with previous M0 or M0.5 prediction files.

In [18]:
# ============================================================
# Cell 18: Save output artifacts
# ============================================================


def json_default(obj: Any) -> Any:
    """JSON serializer for Path, NumPy, and dataclass-friendly objects."""
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, tuple):
        return list(obj)
    return str(obj)


history_df = pd.concat(model_histories, ignore_index=True) if model_histories else pd.DataFrame()

metrics_path = paths.output_dir / "metrics.csv"
leaderboard_path = paths.output_dir / "leaderboard_test.csv"
predictions_specific_path = paths.output_dir / "predictions_val_test_mlp_residual.parquet"
predictions_generic_path = paths.output_dir / "predictions_val_test.parquet"
history_path = paths.output_dir / "training_history.csv"
feature_path = paths.output_dir / "feature_columns_used.json"
notes_path = paths.output_dir / "model_notes.json"
run_config_path = paths.output_dir / "run_config.json"
preprocessor_path = paths.output_dir / "preprocessors.json"

metrics_df.to_csv(metrics_path, index=False, float_format=f"%.{config.float_output_precision}f")
leaderboard_test.to_csv(leaderboard_path, index=False, float_format=f"%.{config.float_output_precision}f")
predictions_df.to_parquet(predictions_specific_path, index=False)
predictions_df.to_parquet(predictions_generic_path, index=False)
history_df.to_csv(history_path, index=False, float_format=f"%.{config.float_output_precision}f")

feature_payload = {
    "manifest_feature_columns": manifest_feature_columns,
    "historical_prior_feature_columns": HISTORICAL_PRIOR_FEATURE_COLUMNS,
    "feature_columns_with_prior": feature_columns_with_prior,
    "label_columns": list(config.label_columns),
}
feature_path.write_text(json.dumps(feature_payload, indent=2, ensure_ascii=False, default=json_default), encoding="utf-8")
notes_path.write_text(json.dumps(model_notes, indent=2, ensure_ascii=False, default=json_default), encoding="utf-8")

preprocessor_payload = {
    "current_preprocessor": current_preprocessor.to_dict(),
    "prior_preprocessor": prior_preprocessor.to_dict(),
    "target_scale": target_scale,
}
preprocessor_path.write_text(json.dumps(preprocessor_payload, indent=2, ensure_ascii=False, default=json_default), encoding="utf-8")

run_payload = {
    "notebook": "FREIGHT-MNet MLP-Residual Notebook",
    "created_at_unix": time.time(),
    "config": asdict(config),
    "paths": asdict(paths),
    "dataset": {
        "n_rows": int(len(df)),
        "n_features_manifest": int(len(manifest_feature_columns)),
        "n_features_with_prior": int(len(feature_columns_with_prior)),
        "label_columns": list(config.label_columns),
        "split_counts": {split: int(mask.sum()) for split, mask in masks.items()},
        "year_counts": {str(k): int(v) for k, v in df[year_col].value_counts().sort_index().items()},
        "origin_column": orig_col,
        "destination_column": dest_col,
        "year_column": year_col,
        "target_scale_minutes": target_scale,
    },
    "package_versions": {
        "python": sys.version,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "torch": torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    },
}
run_config_path.write_text(json.dumps(run_payload, indent=2, ensure_ascii=False, default=json_default), encoding="utf-8")

print("Saved artifacts:")
for p in [
    metrics_path,
    leaderboard_path,
    predictions_specific_path,
    predictions_generic_path,
    history_path,
    feature_path,
    notes_path,
    preprocessor_path,
    run_config_path,
]:
    print(" -", p)

Saved artifacts:
 - E:\NetworkOptimization\pythonProject1\Data\10_experiments\mlp_residual_v1_notebook\east_plus_gulf\metrics.csv
 - E:\NetworkOptimization\pythonProject1\Data\10_experiments\mlp_residual_v1_notebook\east_plus_gulf\leaderboard_test.csv
 - E:\NetworkOptimization\pythonProject1\Data\10_experiments\mlp_residual_v1_notebook\east_plus_gulf\predictions_val_test_mlp_residual.parquet
 - E:\NetworkOptimization\pythonProject1\Data\10_experiments\mlp_residual_v1_notebook\east_plus_gulf\predictions_val_test.parquet
 - E:\NetworkOptimization\pythonProject1\Data\10_experiments\mlp_residual_v1_notebook\east_plus_gulf\training_history.csv
 - E:\NetworkOptimization\pythonProject1\Data\10_experiments\mlp_residual_v1_notebook\east_plus_gulf\feature_columns_used.json
 - E:\NetworkOptimization\pythonProject1\Data\10_experiments\mlp_residual_v1_notebook\east_plus_gulf\model_notes.json
 - E:\NetworkOptimization\pythonProject1\Data\10_experiments\mlp_residual_v1_notebook\east_plus_gulf\preproc

## 19. Optional comparison with the previous M0.5 hybrid baseline

This cell tries to load the previous M0.5 leaderboard if it exists in the standard project path.

The MLP-residual experiment should be interpreted against the strongest M0.5 tabular baselines:

- `residual_lgbm_prior_features`
- `prior_feature_lgbm_direct`

If MLP-residual cannot beat these overall, it may still be useful if it improves stress, sparse-label, IQR, or future cold-OD splits.

In [19]:
# ============================================================
# Cell 19: Optional M0.5 comparison
# ============================================================

previous_m05_leaderboard_path = (
    paths.data_root
    / "10_experiments"
    / "m05_hybrid_baselines_v1_notebook"
    / normalize_scope(config.scope)
    / "leaderboard_test.csv"
)

if previous_m05_leaderboard_path.exists():
    m05_leaderboard = pd.read_csv(previous_m05_leaderboard_path)
    print("Loaded previous M0.5 leaderboard:", previous_m05_leaderboard_path)
    m05_cols = [
        c for c in [
            "rank", "model", "pinball_mean", "weighted_pinball_mean", "mae_q50", "mae_q75",
            "iqr_mae", "stress_top10_mae_q75", "sparse_bottom25_mae_q75",
        ]
        if c in m05_leaderboard.columns
    ]
    print("\nPrevious M0.5 test leaderboard:")
    print(m05_leaderboard[m05_cols].head(10).to_string(index=False))

    print("\nCurrent MLP-residual test leaderboard:")
    print(leaderboard_test[display_columns].to_string(index=False))
else:
    print("Previous M0.5 leaderboard not found. Skipping comparison.")
    print("Expected path:", previous_m05_leaderboard_path)

Loaded previous M0.5 leaderboard: E:\NetworkOptimization\pythonProject1\Data\10_experiments\m05_hybrid_baselines_v1_notebook\east_plus_gulf\leaderboard_test.csv

Previous M0.5 test leaderboard:
                          model  pinball_mean  weighted_pinball_mean    mae_q50     mae_q75    iqr_mae  stress_top10_mae_q75  sparse_bottom25_mae_q75
   residual_lgbm_prior_features    131.187824             116.200505 230.953405  451.623436 384.731525            607.540468               557.575446
      prior_feature_lgbm_direct    131.304712             116.381888 231.615675  449.055148 381.889820            605.877011               554.195459
 residual_lgbm_current_features    132.558496             117.679036 230.401410  465.240711 395.618681            634.736172               573.990126
      hist_lgbm_ensemble_global    146.728685             130.655899 231.257504  462.529352 370.897768            694.091366               562.211623
hist_lgbm_ensemble_per_quantile    148.175706           